In [4]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print("All libraries imported successfully!")
print("Pandas version:", pd.__version__)

Matplotlib is building the font cache; this may take a moment.


All libraries imported successfully!
Pandas version: 1.5.3


In [5]:
# Starting small with just the 2025 season to explore the data and get a feel for it
# Will do basic analysis to confirm with seperate source like PFR that outputs are correct

schedule = nfl.import_schedules([2025])

print(f"Rows: {schedule.shape[0]}, Columns: {schedule.shape[1]}")
schedule.head()

Rows: 285, Columns: 46


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
6991,2025_01_DAL_PHI,2025,REG,1,2025-09-04,Thursday,20:20,DAL,20,PHI,...,11.0,00-0033077,00-0036389,Dak Prescott,Jalen Hurts,Brian Schottenheimer,Nick Sirianni,Shawn Smith,PHI00,Lincoln Financial Field
6992,2025_01_KC_LAC,2025,REG,1,2025-09-05,Friday,20:00,KC,21,LAC,...,NaN,00-0033873,00-0036355,Patrick Mahomes,Justin Herbert,Andy Reid,Jim Harbaugh,Carl Cheffers,LAX01,SoFi Stadium
6993,2025_01_TB_ATL,2025,REG,1,2025-09-07,Sunday,13:00,TB,23,ATL,...,NaN,00-0034855,00-0039917,Baker Mayfield,Michael Penix,Todd Bowles,Raheem Morris,Land Clark,ATL97,Mercedes-Benz Stadium
6994,2025_01_CIN_CLE,2025,REG,1,2025-09-07,Sunday,13:00,CIN,17,CLE,...,10.0,00-0036442,00-0026158,Joe Burrow,Joe Flacco,Zac Taylor,Kevin Stefanski,Adrian Hill,CLE00,FirstEnergy Stadium
6995,2025_01_MIA_IND,2025,REG,1,2025-09-07,Sunday,13:00,MIA,8,IND,...,NaN,00-0036212,00-0035710,Tua Tagovailoa,Daniel Jones,Mike McDaniel,Shane Steichen,Brad Allen,IND00,Lucas Oil Stadium


In [6]:
print(schedule.columns.tolist())

['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [7]:
schedules = nfl.import_schedules(list(range(2014, 2025)))

neededCols = [
    'game_id', 'season', 'game_type', 'week',
    'home_team', 'away_team',
    'home_score', 'away_score',
    'result', 'home_rest', 'away_rest',
    'div_game', 'location'
]

filteredSchedules = schedules[neededCols]

vikingsSchedule = filteredSchedules[(filteredSchedules['home_team'] == 'MIN') | (filteredSchedules['away_team'] == 'MIN')]
print(f"Total Vikings Games: {len(vikingsSchedule)}")
print(vikingsSchedule.head())

Total Vikings Games: 187
              game_id  season game_type  week home_team away_team  home_score  \
3991  2014_01_MIN_STL    2014       REG     1       STL       MIN           6   
4002   2014_02_NE_MIN    2014       REG     2       MIN        NE           7   
4020   2014_03_MIN_NO    2014       REG     3        NO       MIN          20   
4038  2014_04_ATL_MIN    2014       REG     4       MIN       ATL          41   
4042   2014_05_MIN_GB    2014       REG     5        GB       MIN          42   

      away_score  result  home_rest  away_rest  div_game location  
3991          34     -28          7          7         0     Home  
4002          30     -23          7          7         0     Home  
4020           9      11          7          7         0     Home  
4038          28      13          7         10         0     Home  
4042          10      32          4          4         1     Home  


In [8]:
# Tag when the Vikings have a bye week
# If the had_bye column is TRUE, then the Vikings had a bye week in the previous week
vikingsSchedule['had_bye'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['home_rest'] >= 14) |
 (vikingsSchedule['away_team'] == 'MIN') & (vikingsSchedule['away_rest'] >= 14))

#Check to see how many bye games we found
# For 2014 - 2025, we should expect to see 1 bye game per season, so 12 total bye games
print(f"Total Vikings Games with Bye Week: {vikingsSchedule['had_bye'].sum()}")
print(f"Total games without a bye week: {(~vikingsSchedule['had_bye']).sum()}")

Total Vikings Games with Bye Week: 11
Total games without a bye week: 176


C:\Users\johnt\AppData\Local\Temp\ipykernel_17472\3848645147.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vikingsSchedule['had_bye'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['home_rest'] >= 14) |


In [9]:
vikingsSchedule[vikingsSchedule['had_bye'] == True].groupby('season')['had_bye'].count()

season
2014    1
2015    1
2016    1
2017    2
2018    1
2019    1
2020    1
2021    1
2022    1
2024    1
Name: had_bye, dtype: int64

In [10]:
# There are some incorrect values in the bye week count. Adjusting rest day filter to 13, and also filtering for just the regular season
vikingsSchedule = filteredSchedules[((filteredSchedules['home_team'] == 'MIN') | (filteredSchedules['away_team'] == 'MIN')) & (filteredSchedules['game_type'] == 'REG')]
vikingsSchedule['had_bye'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['home_rest'] >= 13) |
 (vikingsSchedule['away_team'] == 'MIN') & (vikingsSchedule['away_rest'] >= 13))

print(f"Total Vikings Games with Bye Week: {vikingsSchedule['had_bye'].sum()}")
print(f"Total games without a bye week: {(~vikingsSchedule['had_bye']).sum()}")
print("Bye Week count by season:")
print(vikingsSchedule[vikingsSchedule['had_bye'] == True].groupby('season')['had_bye'].count())

Total Vikings Games with Bye Week: 11
Total games without a bye week: 169
Bye Week count by season:
season
2014    1
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
Name: had_bye, dtype: int64


C:\Users\johnt\AppData\Local\Temp\ipykernel_17472\3499577600.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vikingsSchedule['had_bye'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['home_rest'] >= 13) |


In [11]:
# Add a column for whether the Vikings won the game or not
vikingsSchedule['vikings_win'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['result'] > 0)) | ((vikingsSchedule['away_team'] == 'MIN') & (vikingsSchedule['result'] < 0))
print(f"# of Vikings Wins: {vikingsSchedule['vikings_win'].sum()}")

# of Vikings Wins: 106


C:\Users\johnt\AppData\Local\Temp\ipykernel_17472\3301999846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vikingsSchedule['vikings_win'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['result'] > 0)) | ((vikingsSchedule['away_team'] == 'MIN') & (vikingsSchedule['result'] < 0))


In [ ]:
# First Question: How often do the Vikings win their first game back from a bye week?

# Find Total Games played, and total games won
totalGames = len(vikingsSchedule)
totalWins = vikingsSchedule['vikings_win'].sum()
overallWinPercentage = totalWins / totalGames if totalGames > 0 else 0
print(f"Total Vikings Games: {totalGames}")
print(f"Total Vikings Wins: {totalWins}")
print(f"Overall Vikings Win Percentage: {overallWinPercentage:.2%}")


postByeGames = vikingsSchedule[vikingsSchedule['had_bye'] == True]
totalByeGames = len(postByeGames)
gamesWon = postByeGames['vikings_win'].sum()
winPercentage = gamesWon / totalByeGames if totalByeGames > 0 else 0
print(f"Total Games After Bye Week: {totalByeGames}")
print(f"Total Games Won After Bye Week: {gamesWon}")
print(f"Vikings Win Percentage After Bye Week: {winPercentage:.2%}")

Total Vikings Games: 180
Total Vikings Wins: 106
Overall Vikings Win Percentage: 58.89%
Total Games After Bye Week: 11
Total Games Won After Bye Week: 5
Vikings Win Percentage After Bye Week: 45.45%
